# OpenAI SDK - Tool Calling MCP

Source: [MCP](https://developers.openai.com/api/docs/guides/tools-connectors-mcp)

There are three kinds of MCP Servers:
- Connectors - OpenAI maintained MCP Server
- Remote MCP Servers - Third parties servers on public internet
- Private / OnPremise MCP Servers - See [Secure MCP Tunnel](https://developers.openai.com/api/docs/guides/secure-mcp-tunnels)

You can write your own MCP server and expose it on internet or via Secure MCP Tunnel

How does it work under the hood:
1. You specify the mcp server / connector in the `tools` parameters
```python
        tools=[
            {
                "type": "mcp",
                "server_label": "deepwiki",
                "server_description": "",
                "server_url": "https://mcp.deepwiki.com/mcp",
                "require_approval": "never",
            },
        ],
```
2. The API retrieves a list of tools and their description from the MCP server
    - the API will put the list into a `mcp_list_tools` item that will be passed to the llm context and added to response.output 
    - In case of errors an `error` field will be added to the mcp_list_tools item
    - Note: the list costs tokens! You can use the `allowed_tools` parameter to retrieve only specific tools (some mcp server may expose a lot of tools)
```python
        McpListTools(
            id='mcpl_0a900b604fbcbb68006a9a9e358c6887d2b65dddfbb1c4be1a', 
            type='mcp_list_tools', 
            server_label='deepwiki',
            error=None 
            tools=[
                    McpListToolsTool(
                        name='ask_question', 
                        description='Ask any question about a GitHub repository and get an AI-powered, context-grounded response.',
                        input_schema={
                            'properties': {
                                'repoName': {...}, 
                                'question': {
                                    'description': 'The question to ask about the repository.', 
                                    'type': 'string'
                                }
                            }, 
                            'required': ['repoName', 'question'], 
                            'type': 'object'}, 
                        annotations={'read_only': False}),  
                    McpListToolsTool(
                        name='read_wiki_contents', 
                        description='View documentation about a GitHub repository.',
                        input_schema={...}, 
                        annotations={...}), 
                    McpListToolsTool(
                        name='read_wiki_structure', 
                        description='Get a list of documentation topics for a GitHub repository.',
                        input_schema={...}, 
                        annotations={...})
            ], 
        )
```
3. If LLM decides to call a mcp tool
    - the API will call the tool with the inputs specified by the model 
    - the API will put the input and output of the call tool into an item `mcp_call` that will be passed to the llm context and added to response.output 
    - In case of errors an `error` field will be added to the mcp_call item
```python
        McpCall(
            id='mcp_0a900b604fbcbb68006a9a9e3886a087d29fb160d90e55dd54', 
            type='mcp_call',
            server_label='deepwiki',
            error=None,
            name='read_wiki_contents', 
            arguments='{"repoName":"modelcontextprotocol/modelcontextprotocol"}', 
            output='# Page: Overview\n\n# Overview\n\n<details>\n<summary>Relevant source files...', 
            approval_request_id=None, 
            status='completed'
        ),
```

Approvals:
1. By default OpenAI will request approval before any data is shared with the mcp tool by creating a `mcp_approval_request` in the response's output
2. You can answer with an `mcp_approval_response`
3. If you want to skip the approval step you can specify it in the mcp declaration
```python
    "require_approval": {
        "never": {"tool_names": ["ask_question", "read_wiki_structure"]}
    },
```

Authentication:
1. Many MCP servers and every connector need OAuth authentication, you can retrieve the oauth token and provide it using the `authorization` field of the mcp tool item

## Connectors

In [3]:
import os
from openai import OpenAI

client = OpenAI()
connector_authorization = os.environ["OPENAI_CONNECTOR_AUTHORIZATION"]

resp = client.responses.create(
    model="gpt-5-nano",
    tools=[
        {
            "type": "mcp",
            "server_label": "Dropbox",
            "connector_id": "connector_dropbox",
            "authorization": connector_authorization,
            "require_approval": "never",
        },
    ],
    input="Summarize the Q2 earnings report.",
)

print(resp.output_text)

BadRequestError: Error code: 400 - {'error': {'message': "Must specify 'authorization' parameter with 'connector_id'.", 'type': 'invalid_request_error', 'param': 'authorization', 'code': None}}

## Remote MCP Servers

In [5]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

resp = client.responses.create(
    model="gpt-5-nano",
    tools=[
        {
            "type": "mcp",
            "server_label": "deepwiki",
            "server_description": "",
            "server_url": "https://mcp.deepwiki.com/mcp",
            "require_approval": "never",
        },
    ],
    input="What transport protocols does the 2025-03-26 version of the MCP spec (modelcontextprotocol/modelcontextprotocol) support?",
)

print(f"usage: {resp.usage}")
print(f"output_text: {resp.output_text}")

usage: ResponseUsage(input_tokens=117056, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=1248, output_tokens_details=OutputTokensDetails(reasoning_tokens=1088), total_tokens=118304, compute_units=None)
output_text: Two transports:

- stdio (local subprocess transport)  
  Messages flow over the process’s stdin/stdout using newline-delimited JSON-RPC.

- Streamable HTTP (remote transport)  
  Messages are sent via HTTP POST, with optional SSE streaming for notifications.


### Response.output example

```python
Response(
    ...
    output=[
        McpListTools(
            id='mcpl_0a900b604fbcbb68006a9a9e358c6887d2b65dddfbb1c4be1a', 
            server_label='deepwiki', 
            tools=[
                    McpListToolsTool(
                        name='ask_question', 
                        description='Ask any question about a GitHub repository and get an AI-powered, context-grounded response.',
                        input_schema={
                            'properties': {
                                'repoName': { 
                                    'description': 'GitHub repository or list of repositories (max 10) in owner/repo format.',
                                    'anyOf': [{'type': 'string'}, {'items': {'type': 'string'}, 'type': 'array'}]
                                }, 
                                'question': {
                                    'description': 'The question to ask about the repository.', 
                                    'type': 'string'
                                }
                            }, 
                            'required': ['repoName', 'question'], 
                            'type': 'object'}, 
                        annotations={'read_only': False}),  
                    McpListToolsTool(
                        name='read_wiki_contents', 
                        description='View documentation about a GitHub repository.',
                        input_schema={...}, 
                        annotations={...}), 
                    McpListToolsTool(
                        name='read_wiki_structure', 
                        description='Get a list of documentation topics for a GitHub repository.',
                        input_schema={...}, 
                        annotations={...})], 
            type='mcp_list_tools', 
            error=None
        ), 
        ResponseReasoningItem(...), 
        McpCall(
            id='mcp_0a900b604fbcbb68006a9a9e3886a087d29fb160d90e55dd54', 
            arguments='{"repoName":"modelcontextprotocol/modelcontextprotocol"}', 
            name='read_wiki_contents', 
            server_label='deepwiki', 
            type='mcp_call', 
            approval_request_id=None, 
            error=None,
            output='# Page: Overview\n\n# Overview\n\n<details>\n<summary>Relevant source files...', 
            status='completed'
        ), 
        ResponseReasoningItem(...), 
        ResponseOutputMessage(
            id='msg_0a900b604fbcbb68006a9a9e427ed487d2985ea71251f5d969', 
            content=[
                ResponseOutputText(annotations=[], 
                                   text='Two transports:\n\n- stdio (local subprocess transport)  \n  Messages flow...', 
                                   type='output_text', 
                                   logprobs=[])
            ], 
            role='assistant', 
            status='completed', 
            type='message', 
            phase=None)
    ], 
...
)
```